In [1]:
pip install git+https://github.com/elmoiv/azapi.git

  Cloning https://github.com/elmoiv/azapi.git to c:\users\16172\appdata\local\temp\pip-req-build-xzm0c4lk
  Resolved https://github.com/elmoiv/azapi.git to commit ef28e5a06d23690cef6f3c72f043f1ea55bf7858
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/elmoiv/azapi.git 'C:\Users\16172\AppData\Local\Temp\pip-req-build-xzm0c4lk'


In [2]:
# Take a test
import re
import requests
from bs4 import BeautifulSoup

def fetch_azlyrics(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
    }
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")


    main = soup.select_one("div.col-xs-12.col-lg-8.text-center")
    divs = main.find_all("div") if main else soup.find_all("div")

    lyrics = None
    for d in divs:
        if d.get("class") is None and d.get("id") is None:
            text = d.get_text("\n", strip=True)

            if len(text.splitlines()) >= 5 and len(text) > 200:
                lyrics = text
                break

    if not lyrics:
        raise RuntimeError("Failed to locate lyrics div (page structure may have changed).")

    lyrics = re.sub(r"\n{3,}", "\n\n", lyrics).strip()
    return lyrics

url = "https://www.azlyrics.com/lyrics/taylorswift/blankspace.html"
lyrics = fetch_azlyrics(url)
print(lyrics[:800])


Nice to meet you, where you been?
I could show you incredible things
Magic, madness, heaven, sin
Saw you there, and I thought
"Oh my God, look at that face
You look like my next mistake
Love's a game, wanna play?"
Ayy
New money, suit and tie
I can read you like a magazine
Ain't it funny? Rumors fly
And I know you heard about me
So hey, let's be friends
I'm dying to see how this one ends
Grab your passport and my hand
I can make the bad guys good for a weekend
So it's gonna be forever
Or it's gonna go down in flames
You can tell me when it's over, mm
If the high was worth the pain
Got a long list of ex-lovers
They'll tell you I'm insane
'Cause you know I love the players
And you love the game
'Cause we're young and we're reckless
We'll take this way too far
It'll leave you breathless, hmm
O


In [ ]:
import os
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup



UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
]

def make_headers():
    """Generate headers with a randomized (but realistic) User-Agent."""
    return {
        "User-Agent": random.choice(UA_POOL) + " (Academic research; contact: liyou@uchicago.edu)",
        "Accept-Language": "en-US,en;q=0.9",
    }

# ============================================================
# 1. Fetch lyrics from AZLyrics
# ============================================================

def fetch_azlyrics(url: str, headers: dict, timeout: int = 20) -> str:
    r = requests.get(url, headers=headers, timeout=timeout)


    if r.status_code in (403, 429):
        raise RuntimeError(f"Blocked by AZLyrics (HTTP {r.status_code})")

    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")


    main = soup.select_one("div.col-xs-12.col-lg-8.text-center")
    divs = main.find_all("div") if main else soup.find_all("div")

    lyrics = None
    for d in divs:
        if d.get("class") is None and d.get("id") is None:
            text = d.get_text("\n", strip=True)
            if len(text.splitlines()) >= 5 and len(text) > 200:
                lyrics = text
                break

    if not lyrics:
        raise RuntimeError("Failed to locate lyrics div")

    lyrics = re.sub(r"\n{3,}", "\n\n", lyrics).strip()
    return lyrics

# ============================================================
# 2. URL construction 
# ============================================================

def extract_main_artist(artist: str) -> str:
    return artist.split(",")[0].strip() if artist else ""

def normalize_simple(s: str) -> str:
    s = s.lower()
    return re.sub(r"[^a-z0-9]", "", s)

def build_azlyrics_url(artist: str, title: str) -> str:
    artist_main = extract_main_artist(artist)
    return (
        "https://www.azlyrics.com/lyrics/"
        f"{normalize_simple(artist_main)}/"
        f"{normalize_simple(title)}.html"
    )

In [ ]:
BATCH_SIZE  = 50
MAX_BATCHES = 2   # Number of batches to run automatically per execution


# Update the file paths below

INPUT_CSV = r"C:\Users\16172\OneDrive\桌面\RA\Music&Finance\uncovered.csv"
OUT_DIR   = r"C:\Users\16172\OneDrive\桌面\RA\Music&Finance\lyrics_batches"

os.makedirs(OUT_DIR, exist_ok=True)

PROGRESS_CSV = os.path.join(OUT_DIR, "covered_with_lyrics_progress.csv")

# ============================================================
# Load data (resume-safe)
# ============================================================

if os.path.exists(PROGRESS_CSV):
    df = pd.read_csv(PROGRESS_CSV, encoding="utf-8-sig")
    print("Loaded progress:", PROGRESS_CSV)
else:
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    print("Loaded input:", INPUT_CSV)

for col in ["azlyrics_url", "lyrics", "lyrics_status"]:
    if col not in df.columns:
        df[col] = ""

# ============================================================
# Automatic multi-batch crawling
# ============================================================

batch_counter = 0

while True:
    status = df["lyrics_status"].fillna("").astype(str).str.strip()
    todo_idx = df.index[status == ""]

    remaining = len(todo_idx)
    print(f"\nRemaining rows to process: {remaining}")

    if remaining == 0:
        print("All done")
        break

    if batch_counter >= MAX_BATCHES:
        print(f"Reached MAX_BATCHES={MAX_BATCHES}. Stop here.")
        break

    batch_idx = todo_idx[:BATCH_SIZE]
    batch_counter += 1

    completed = int((df["lyrics_status"] != "").sum())
    batch_no = (completed // BATCH_SIZE) + 1
    batch_file = os.path.join(OUT_DIR, f"batch{batch_no:02d}.csv")

    print(f"Processing batch{batch_no:02d} | rows {batch_idx[0]} ... {batch_idx[-1]}")

    for j, idx in enumerate(batch_idx, start=1):
        artist = str(df.loc[idx, "artist_name"]).strip()
        title  = str(df.loc[idx, "track_name"]).strip()

        url = ""
        try:
            url = build_azlyrics_url(artist, title)
            headers = make_headers()
            lyrics = fetch_azlyrics(url, headers=headers)

            df.at[idx, "azlyrics_url"] = url
            df.at[idx, "lyrics"] = lyrics
            df.at[idx, "lyrics_status"] = "ok"

        except Exception as e:
            df.at[idx, "azlyrics_url"] = url
            df.at[idx, "lyrics"] = ""
            df.at[idx, "lyrics_status"] = f"fail: {type(e).__name__}"

        # Per request: random sleep of 4–8 seconds
        time.sleep(random.uniform(4, 9))

        # Every 10 requests: take an additional longer break
        if j % 10 == 0:
            print(f"Processed {j}/{len(batch_idx)} in this batch, taking a long break...")
            time.sleep(random.uniform(18, 29))

    # Save current batch results
    df.loc[batch_idx].to_csv(batch_file, index=False, encoding="utf-8-sig")
    print("Saved batch result to:", batch_file)

    # Save overall progress (for resume-safe execution)
    df.to_csv(PROGRESS_CSV, index=False, encoding="utf-8-sig")
    print("Saved progress to:", PROGRESS_CSV)

    # Extra long break between batches (safer)
    # time.sleep(random.uniform(45, 70))
